# Chapter 11: Partial Differential Equations

## The Wave Equation

### Python Simulation of the Wave Equation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def wave_equation_1d(L, T, c, nx, nt, f_init, g_init, bc_left=0, bc_right=0):
    """
    Solve 1D wave equation u_tt = c^2 * u_xx using leapfrog finite differences.

    Parameters:
    -----------
    L : float
        Domain length
    T : float
        Total time
    c : float
        Wave speed
    nx : int
        Number of spatial points
    nt : int
        Number of time steps
    f_init : callable
        Initial displacement u(x, 0)
    g_init : callable
        Initial velocity u_t(x, 0)
    bc_left, bc_right : float
        Boundary conditions at x=0 and x=L

    Returns:
    --------
    x : ndarray
        Spatial grid
    t : ndarray
        Time grid
    u : ndarray
        Solution array of shape (nt, nx)
    """
    dx = L / (nx - 1)
    dt = T / (nt - 1)
    r = c * dt / dx  # Courant number

    assert r <= 1.0, f"CFL violation: r = {r} > 1; use smaller dt or larger dx"

    x = np.linspace(0, L, nx)
    t = np.linspace(0, T, nt)
    u = np.zeros((nt, nx))

    # Initial displacement
    u[0, :] = f_init(x)

    # First time step using initial velocity
    for i in range(1, nx - 1):
        u[1, i] = (0.5 * (r**2) * (u[0, i+1] - 2*u[0, i] + u[0, i-1]) +
                   u[0, i] + dt * g_init(x[i]))

    # Boundary conditions
    u[:, 0] = bc_left
    u[:, -1] = bc_right

    # Leapfrog time stepping
    for n in range(1, nt - 1):
        for i in range(1, nx - 1):
            u[n+1, i] = ((r**2) * (u[n, i+1] - 2*u[n, i] + u[n, i-1]) +
                         2*u[n, i] - u[n-1, i])

    return x, t, u

# Example: plucked string (triangular initial displacement)
L = 1.0
T = 2.0
c = 1.0
nx = 101
nt = 201

def f_init(x):
    """Triangular initial shape: plucked at center."""
    return np.where(x <= 0.5, 2*x, 2*(1-x))

def g_init(x):
    """Released from rest."""
    return np.zeros_like(x)

x, t, u = wave_equation_1d(L, T, c, nx, nt, f_init, g_init)

# Visualization: solution at selected times
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
times = [0, T/4, T/2, 3*T/4]
for idx, (ax, t_target) in enumerate(zip(axes.flat, times)):
    t_idx = int(t_target * (nt - 1) / T)
    ax.plot(x, u[t_idx, :], 'b-', linewidth=2)
    ax.fill_between(x, 0, u[t_idx, :], alpha=0.3)
    ax.set_xlabel('Position x')
    ax.set_ylabel('Displacement u(x, t)')
    ax.set_title(f'Wave Equation: $t = {t[t_idx]:.2f}$')
    ax.set_ylim([-1.1, 1.1])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## The Heat Equation (Diffusion)

### Numerical Solution: Explicit and Implicit Schemes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

class HeatEquationSolver:
    """Solvers for the 1D heat equation."""

    def __init__(self, L, T, alpha, nx, nt):
        self.L = L
        self.T = T
        self.alpha = alpha
        self.nx = nx
        self.nt = nt
        self.dx = L / (nx - 1)
        self.dt = T / (nt - 1)
        self.r = alpha * self.dt / (self.dx**2)
        self.x = np.linspace(0, L, nx)
        self.t = np.linspace(0, T, nt)

    def explicit_scheme(self, f_init, bc_left=0, bc_right=0):
        """FTCS explicit scheme."""
        if self.r > 0.5:
            print(f"Warning: r={self.r:.4f} > 0.5; explicit scheme unstable")

        u = np.zeros((self.nt, self.nx))
        u[0, :] = f_init(self.x)

        for n in range(self.nt - 1):
            u[n+1, 1:-1] = (self.r * u[n, 2:] + (1 - 2*self.r) * u[n, 1:-1] +
                            self.r * u[n, :-2])
            u[n+1, 0] = bc_left
            u[n+1, -1] = bc_right

        return u

    def implicit_scheme(self, f_init, bc_left=0, bc_right=0):
        """Crank-Nicolson implicit scheme."""
        u = np.zeros((self.nt, self.nx))
        u[0, :] = f_init(self.x)

        # Tridiagonal matrix
        diag_lower = -0.5 * self.r * np.ones(self.nx - 3)
        diag_main = (1 + self.r) * np.ones(self.nx - 2)
        diag_upper = -0.5 * self.r * np.ones(self.nx - 3)

        A = diags([diag_lower, diag_main, diag_upper],
                  [-1, 0, 1], shape=(self.nx - 2, self.nx - 2), format='csr')

        for n in range(self.nt - 1):
            rhs = ((1 - self.r) * u[n, 1:-1] +
                   0.5 * self.r * (u[n, 2:] + u[n, :-2]))
            rhs[0] += 0.5 * self.r * bc_left
            rhs[-1] += 0.5 * self.r * bc_right
            u[n+1, 1:-1] = spsolve(A, rhs)
            u[n+1, 0] = bc_left
            u[n+1, -1] = bc_right

        return u

# Test case (nt chosen so that r = 0.25 satisfies the FTCS limit r <= 1/2)
L, T, alpha = 1.0, 0.2, 0.1
nx, nt = 51, 201

def f_init(x):
    return np.exp(-50*(x - 0.5)**2)  # Gaussian bump

solver = HeatEquationSolver(L, T, alpha, nx, nt)
u_explicit = solver.explicit_scheme(f_init)
u_implicit = solver.implicit_scheme(f_init)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_indices = [0, nt//4, nt//2, 3*nt//4, nt-1]
colors = plt.cm.viridis(np.linspace(0, 1, len(t_indices)))

for idx, t_idx in enumerate(t_indices):
    axes[0].plot(solver.x, u_explicit[t_idx, :], color=colors[idx],
                 label=f't={solver.t[t_idx]:.3f}', linewidth=1.5)
    axes[1].plot(solver.x, u_implicit[t_idx, :], color=colors[idx],
                 label=f't={solver.t[t_idx]:.3f}', linewidth=1.5)

axes[0].set_title('Explicit FTCS')
axes[1].set_title('Implicit Crank-Nicolson')
for ax in axes:
    ax.set_xlabel('Position x')
    ax.set_ylabel('Temperature u(x, t)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Laplace's Equation

### Numerical Solution by Finite Differences

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def laplace_solver_rectangular(a, b, nx, ny, bc_top, bc_bottom, bc_left, bc_right):
    """
    Solve Laplace equation on rectangle [0,a] x [0,b] with Dirichlet BCs.

    bc_* : callable(s) or array
        Boundary conditions on top, bottom, left, right edges.
    """
    hx = a / (nx - 1)
    hy = b / (ny - 1)
    x = np.linspace(0, a, nx)
    y = np.linspace(0, b, ny)

    # Jacobi iteration on the 5-point stencil
    u = np.zeros((ny, nx))

    # Set boundary conditions
    if callable(bc_top):
        u[-1, :] = bc_top(x)
    else:
        u[-1, :] = bc_top

    if callable(bc_bottom):
        u[0, :] = bc_bottom(x)
    else:
        u[0, :] = bc_bottom

    if callable(bc_left):
        u[:, 0] = bc_left(y)
    else:
        u[:, 0] = bc_left

    if callable(bc_right):
        u[:, -1] = bc_right(y)
    else:
        u[:, -1] = bc_right

    # Jacobi iteration (vectorized): every interior point is updated
    # from the PREVIOUS iterate u_old, which is what defines Jacobi
    for iteration in range(10000):
        u_old = u.copy()
        u[1:-1, 1:-1] = 0.25 * (u_old[2:, 1:-1] + u_old[:-2, 1:-1] +
                                u_old[1:-1, 2:] + u_old[1:-1, :-2])

        # Check convergence
        if np.max(np.abs(u - u_old)) < 1e-6:
            print(f"Converged after {iteration + 1} iterations")
            break

    return x, y, u

# Test case
a, b = 1.0, 1.0
nx, ny = 51, 51
x, y, u = laplace_solver_rectangular(
    a, b, nx, ny,
    bc_top=lambda x: np.sin(np.pi*x),
    bc_bottom=0.0,
    bc_left=0.0,
    bc_right=0.0
)

# Visualization
X, Y = np.meshgrid(x, y)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.contourf(X, Y, u, levels=20, cmap='RdBu_r')
ax.contour(X, Y, u, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title("Solution to Laplace's Equation on Rectangle")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Solution of PDEs with Laplace Transforms

### Python SymPy Verification

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc

# Symbolic form of the Laplace-domain solution
x, t, s, alpha, Q, kappa = sp.symbols('x t s alpha Q kappa', real=True, positive=True)

beta = sp.sqrt(s / alpha)

# Solution to heat equation on the semi-infinite domain
# hat_u(x, s) = (Q/(kappa*s*beta)) * exp(-beta*x)
hat_u = (Q / (kappa * s * beta)) * sp.exp(-beta * x)

print("Laplace domain solution for semi-infinite heat rod:")
print(f"hat_u(x, s) = {hat_u}")

# Numerical solution: compare analytical and numerical
def analytical_solution(x_val, t_val, alpha_val=0.1, Q_val=1, kappa_val=1):
    """Analytical solution from Laplace transform inversion."""
    if t_val < 1e-10:
        return 0
    sqrt_at = np.sqrt(alpha_val * t_val)
    term1 = 2 * sqrt_at / np.sqrt(np.pi) * np.exp(-x_val**2 / (4 * alpha_val * t_val))
    term2 = x_val * erfc(x_val / (2 * sqrt_at))
    return (Q_val / kappa_val) * (term1 - term2)

# Plot
x_vals = np.linspace(0, 3, 100)
t_vals = [0.01, 0.05, 0.1, 0.2]
fig, ax = plt.subplots(figsize=(10, 6))

for t_val in t_vals:
    u_vals = [analytical_solution(x_v, t_val) for x_v in x_vals]
    ax.plot(x_vals, u_vals, label=f't = {t_val}', linewidth=2)

ax.set_xlabel('Position x')
ax.set_ylabel('Temperature u(x, t)')
ax.set_title('Heat Diffusion from Constant Heat Flux at x=0')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Solution of PDEs with Fourier Transforms

### Python Computation of Fourier Solution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import special

# Analytical solution: heat equation with Gaussian initial condition
def heat_solution_fourier(x, t, alpha, A, lambda_param):
    """
    Solution to heat equation with Gaussian initial condition.
    u(x, 0) = A * exp(-lambda * x^2)
    """
    denominator = 1 + 4 * lambda_param * alpha * t
    amplitude = A / np.sqrt(denominator)
    exponent = -lambda_param * x**2 / denominator
    return amplitude * np.exp(exponent)

# Parameters
alpha = 0.1
A = 1.0
lambda_param = 1.0
t_vals = [0, 0.05, 0.1, 0.2, 0.5, 1.0]
x_vals = np.linspace(-3, 3, 200)

# Plot solutions at different times
fig, ax = plt.subplots(figsize=(10, 6))
for t in t_vals:
    u_vals = heat_solution_fourier(x_vals, t, alpha, A, lambda_param)
    label = f't = {t}' if t > 0 else 't = 0 (initial)'
    ax.plot(x_vals, u_vals, linewidth=2, label=label)

ax.set_xlabel('Position x')
ax.set_ylabel('Temperature u(x, t)')
ax.set_title('Heat Diffusion: Gaussian Initial Condition')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Verify Fourier transform relationship
# Compute initial condition Fourier transform
def gaussian_fourier(k, A, lambda_param):
    """Fourier transform of Gaussian: A*exp(-lambda*x^2)."""
    return A * np.sqrt(np.pi / lambda_param) * np.exp(-k**2 / (4 * lambda_param))

# Plot in k-domain
k_vals = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Time-domain solutions
times = [0, 0.1, 0.5]
for i, t in enumerate(times):
    u_vals = heat_solution_fourier(x_vals, t, alpha, A, lambda_param)
    axes[i].plot(x_vals, u_vals, 'b-', linewidth=2)
    axes[i].fill_between(x_vals, 0, u_vals, alpha=0.3)
    axes[i].set_title(f'Position domain, t = {t}')
    axes[i].set_xlabel('x')
    axes[i].set_ylabel('u(x, t)')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim([-3, 3])

plt.tight_layout()
plt.show()

# Summary: variance growth
print("Width of Gaussian (measure of diffusion):")
for t in [0, 0.1, 0.5, 1.0]:
    width = np.sqrt((1 + 4 * lambda_param * alpha * t) / (2 * lambda_param))
    print(f"  t = {t:4.1f}: width ~ {width:.4f}")

## Numerical Methods for PDEs

### Python Implementation: Complete Schemes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# ============= 1. FTCS Explicit Scheme (Heat Equation) =============
class HeatEquationFTCS:
    def __init__(self, L, T, alpha, nx, nt):
        self.L, self.T, self.alpha = L, T, alpha
        self.nx, self.nt = nx, nt
        self.dx = L / (nx - 1)
        self.dt = T / (nt - 1)
        self.r = alpha * self.dt / (self.dx ** 2)

        if self.r > 0.5:
            print(f"Warning: r = {self.r:.4f} > 0.5; scheme is unstable!")

        self.x = np.linspace(0, L, nx)
        self.t = np.linspace(0, T, nt)

    def solve(self, f_init, bc_left=0, bc_right=0):
        u = np.zeros((self.nt, self.nx))
        u[0, :] = f_init(self.x)

        for n in range(self.nt - 1):
            u[n+1, 1:-1] = (self.r * u[n, 2:] + (1 - 2*self.r) * u[n, 1:-1] +
                            self.r * u[n, :-2])
            u[n+1, 0] = bc_left
            u[n+1, -1] = bc_right

        return u

# ============= 2. Crank--Nicolson Implicit Scheme =============
class HeatEquationCN:
    def __init__(self, L, T, alpha, nx, nt):
        self.L, self.T, self.alpha = L, T, alpha
        self.nx, self.nt = nx, nt
        self.dx = L / (nx - 1)
        self.dt = T / (nt - 1)
        self.r = alpha * self.dt / (self.dx ** 2)

        self.x = np.linspace(0, L, nx)
        self.t = np.linspace(0, T, nt)

    def solve(self, f_init, bc_left=0, bc_right=0):
        u = np.zeros((self.nt, self.nx))
        u[0, :] = f_init(self.x)

        # Build tridiagonal system matrix
        diag_l = -0.5 * self.r * np.ones(self.nx - 3)
        diag_m = (1 + self.r) * np.ones(self.nx - 2)
        diag_u = -0.5 * self.r * np.ones(self.nx - 3)
        A = diags([diag_l, diag_m, diag_u], [-1, 0, 1],
                  shape=(self.nx - 2, self.nx - 2), format='csr')

        for n in range(self.nt - 1):
            rhs = ((1 - self.r) * u[n, 1:-1] +
                   0.5 * self.r * (u[n, 2:] + u[n, :-2]))
            rhs[0] += 0.5 * self.r * bc_left
            rhs[-1] += 0.5 * self.r * bc_right
            u[n+1, 1:-1] = spsolve(A, rhs)
            u[n+1, 0] = bc_left
            u[n+1, -1] = bc_right

        return u

# ============= 3. Leapfrog Scheme (Wave Equation) =============
class WaveEquationLeapfrog:
    def __init__(self, L, T, c, nx, nt):
        self.L, self.T, self.c = L, T, c
        self.nx, self.nt = nx, nt
        self.dx = L / (nx - 1)
        self.dt = T / (nt - 1)
        self.nu = c * self.dt / self.dx

        if self.nu > 1.0:
            print(f"Warning: CFL violated: nu = {self.nu:.4f} > 1.0; scheme unstable!")

        self.x = np.linspace(0, L, nx)
        self.t = np.linspace(0, T, nt)

    def solve(self, f_init, g_init, bc_left=0, bc_right=0):
        u = np.zeros((self.nt, self.nx))
        u[0, :] = f_init(self.x)

        # First time step
        for i in range(1, self.nx - 1):
            u[1, i] = (u[0, i] + self.dt * g_init(self.x[i]) +
                       0.5 * (self.nu ** 2) * (u[0, i+1] - 2*u[0, i] + u[0, i-1]))
        u[1, 0] = bc_left
        u[1, -1] = bc_right

        # Leapfrog iterations
        for n in range(1, self.nt - 1):
            for i in range(1, self.nx - 1):
                u[n+1, i] = (2*u[n, i] - u[n-1, i] +
                             (self.nu ** 2) * (u[n, i+1] - 2*u[n, i] + u[n, i-1]))
            u[n+1, 0] = bc_left
            u[n+1, -1] = bc_right

        return u

# ============= 4. Jacobi Iteration (Laplace's Equation) =============
class LaplaceJacobi:
    def __init__(self, a, b, nx, ny, tol=1e-6, max_iter=10000):
        self.a, self.b = a, b
        self.nx, self.ny = nx, ny
        self.hx = a / (nx - 1)
        self.hy = b / (ny - 1)
        self.tol = tol
        self.max_iter = max_iter

        self.x = np.linspace(0, a, nx)
        self.y = np.linspace(0, b, ny)

    def solve(self, bc_top, bc_bottom, bc_left, bc_right):
        u = np.zeros((self.ny, self.nx))

        # Set boundary conditions
        u[-1, :] = bc_top(self.x) if callable(bc_top) else bc_top
        u[0, :] = bc_bottom(self.x) if callable(bc_bottom) else bc_bottom
        u[:, 0] = bc_left(self.y) if callable(bc_left) else bc_left
        u[:, -1] = bc_right(self.y) if callable(bc_right) else bc_right

        for iteration in range(self.max_iter):
            u_old = u.copy()
            u[1:-1, 1:-1] = 0.25 * (u_old[2:, 1:-1] + u_old[:-2, 1:-1] +
                                    u_old[1:-1, 2:] + u_old[1:-1, :-2])

            if np.max(np.abs(u - u_old)) < self.tol:
                print(f"Converged in {iteration + 1} iterations")
                break

        return u

# ============= Test All Schemes =============
if __name__ == "__main__":
    # Heat equation: test FTCS vs Crank--Nicolson
    print("Heat Equation: FTCS vs Crank--Nicolson")
    L, T, alpha, nx, nt = 1.0, 0.1, 0.1, 51, 51
    f_init = lambda x: np.exp(-100*(x - 0.5)**2)

    solver_ftcs = HeatEquationFTCS(L, T, alpha, nx, nt)
    u_ftcs = solver_ftcs.solve(f_init)

    solver_cn = HeatEquationCN(L, T, alpha, nx, nt)
    u_cn = solver_cn.solve(f_init)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    times = [0, nt//4, nt//2, 3*nt//4, nt-1]
    colors = plt.cm.viridis(np.linspace(0, 1, len(times)))

    for idx, t_idx in enumerate(times):
        axes[0].plot(solver_ftcs.x, u_ftcs[t_idx, :], color=colors[idx],
                     label=f't={solver_ftcs.t[t_idx]:.3f}', linewidth=1.5)
        axes[1].plot(solver_cn.x, u_cn[t_idx, :], color=colors[idx],
                     label=f't={solver_cn.t[t_idx]:.3f}', linewidth=1.5)

    axes[0].set_title('FTCS (Explicit)')
    axes[1].set_title('Crank--Nicolson (Implicit)')
    for ax in axes:
        ax.set_xlabel('Position x')
        ax.set_ylabel('Temperature u(x, t)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Wave equation: plucked string
    print("\nWave Equation: Plucked String")
    L, T, c, nx, nt = 1.0, 2.0, 1.0, 101, 201
    f_wave = lambda x: np.where(x <= 0.5, 2*x, 2*(1-x))
    g_wave = lambda x: np.zeros_like(x)

    solver_wave = WaveEquationLeapfrog(L, T, c, nx, nt)
    u_wave = solver_wave.solve(f_wave, g_wave)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    times = [0, nt//4, nt//2, 3*nt//4]
    for idx, (ax, t_idx) in enumerate(zip(axes.flat, times)):
        ax.plot(solver_wave.x, u_wave[t_idx, :], 'b-', linewidth=2)
        ax.fill_between(solver_wave.x, 0, u_wave[t_idx, :], alpha=0.3)
        ax.set_title(f't = {solver_wave.t[t_idx]:.2f}')
        ax.set_xlabel('Position x')
        ax.set_ylabel('Displacement u(x, t)')
        ax.set_ylim([-1.1, 1.1])
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Laplace's equation: Jacobi iteration
    print("\nLaplace's Equation: Jacobi Iteration")
    a, b, nx, ny = 1.0, 1.0, 51, 51
    solver_laplace = LaplaceJacobi(a, b, nx, ny)
    u_laplace = solver_laplace.solve(
        bc_top=lambda x: np.sin(np.pi*x),
        bc_bottom=0.0,
        bc_left=0.0,
        bc_right=0.0
    )

    X, Y = np.meshgrid(solver_laplace.x, solver_laplace.y)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.contourf(X, Y, u_laplace, levels=20, cmap='RdBu_r')
    ax.contour(X, Y, u_laplace, levels=10, colors='black', alpha=0.3, linewidths=0.5)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title("Laplace's Equation (Jacobi)")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## Applications in Engineering

### Python Case Studies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============= Case Study 1: Heat Conduction in Composite Rod =============
# Two-layer rod, FTCS in each layer with a simple interface treatment.
# dt is chosen so that r = alpha*dt/dx^2 <= 1/2 in BOTH layers.
L1, L2 = 0.5, 0.5
L_total = L1 + L2
alpha1, alpha2 = 0.1, 0.05
nx, nt = 101, 2501
dx = L_total / (nx - 1)
dt = 4e-4                      # r1 = 0.4, r2 = 0.2 (stable)
T_left, T_right = 100, 0

x = np.linspace(0, L_total, nx)
u = np.zeros((nt, nx))
u[:, 0] = T_left
u[:, -1] = T_right
u[0, 1:-1] = 0.0               # rod initially cold; heated from the left

idx_interface = int(L1 / dx)
r1 = alpha1 * dt / dx**2
r2 = alpha2 * dt / dx**2
print(f"Stability numbers: r1 = {r1:.2f}, r2 = {r2:.2f}")

for n in range(nt - 1):
    # Layer 1
    u[n+1, 1:idx_interface] = (r1 * u[n, 2:idx_interface+1] +
                               (1 - 2*r1) * u[n, 1:idx_interface] +
                               r1 * u[n, :idx_interface-1])

    # Layer 2
    u[n+1, idx_interface+1:-1] = (r2 * u[n, idx_interface+2:] +
                                   (1 - 2*r2) * u[n, idx_interface+1:-1] +
                                   r2 * u[n, idx_interface:-2])

    # Interface: continuity (simplified)
    u[n+1, idx_interface] = 0.5 * (u[n+1, idx_interface-1] + u[n+1, idx_interface+1])

fig, ax = plt.subplots(figsize=(10, 6))
times = [nt//50, nt//8, nt//4, nt//2, nt-1]
colors = plt.cm.viridis(np.linspace(0, 1, len(times)))

for idx, t_idx in enumerate(times):
    ax.plot(x, u[t_idx, :], color=colors[idx], label=f't={t_idx*dt:.3f} s', linewidth=2)
ax.axvline(x[idx_interface], color='red', linestyle='--', alpha=0.5, linewidth=1,
           label='interface')

ax.set_xlabel('Position x (m)')
ax.set_ylabel(r'Temperature ($^\circ$C)')
ax.set_title('Heat Conduction in Composite Rod (alpha1=0.1, alpha2=0.05)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============= Case Study 2: Advection--Diffusion of Pollutant =============
def advection_diffusion_1d(L, T, u_vel, D, nx, nt, c_init_type='gaussian'):
    """
    Numerical solution of advection--diffusion equation:
    dc/dt + u*dc/dx = D*d2c/dx2
    """
    dx = L / (nx - 1)
    dt = T / (nt - 1)
    x = np.linspace(0, L, nx)
    t = np.linspace(0, T, nt)

    c = np.zeros((nt, nx))

    # Initial condition
    if c_init_type == 'gaussian':
        c[0, :] = 100 * np.exp(-100 * (x - 0.3)**2)
    elif c_init_type == 'square':
        c[0, (x >= 0.2) & (x <= 0.4)] = 100

    # Boundary conditions
    c[:, 0] = 0  # Zero at left
    c[:, -1] = 0  # Zero at right

    # FTCS scheme (explicit)
    Pe = u_vel * dx / D  # Peclet number
    r_diff = D * dt / dx**2
    r_adv = u_vel * dt / (2 * dx)

    print(f"Peclet number: {Pe:.2f}")
    print(f"Diffusion number: {r_diff:.4f}")
    print(f"Advection number: {r_adv:.4f}")

    for n in range(nt - 1):
        c[n+1, 1:-1] = (c[n, 1:-1] -
                        r_adv * (c[n, 2:] - c[n, :-2]) +
                        r_diff * (c[n, 2:] - 2*c[n, 1:-1] + c[n, :-2]))

    return x, t, c

# Parameters: dx = 0.01, dt = 0.0025 give a diffusion number of 0.25
# and a Courant number of 0.25, both within the explicit stability limits.
L, T = 1.0, 0.5
u_vel = 1.0
D = 0.01
nx, nt = 101, 201

x_ad, t_ad, c_ad = advection_diffusion_1d(L, T, u_vel, D, nx, nt, 'gaussian')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
times_idx = [0, nt//3, 2*nt//3, nt-1]

for idx, (ax, t_idx) in enumerate(zip(axes.flat, times_idx)):
    ax.plot(x_ad, c_ad[t_idx, :], 'b-', linewidth=2)
    ax.fill_between(x_ad, 0, c_ad[t_idx, :], alpha=0.3)
    ax.set_xlabel('Position x (m)')
    ax.set_ylabel('Concentration c (mg/L)')
    ax.set_title(f'Pollutant Transport, t = {t_ad[t_idx]:.2f}')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 110])

plt.suptitle(f'Advection-Diffusion: u={u_vel}, D={D}, Pe={u_vel*L/D:.1f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nPollutant centroid at t=0: x = {np.sum(x_ad * c_ad[0,:]) / np.sum(c_ad[0,:]):.3f}")
print(f"Pollutant centroid at t=T: x = {np.sum(x_ad * c_ad[-1,:]) / np.sum(c_ad[-1,:]):.3f}")
print(f"Expected advection distance: {u_vel * T:.3f}")

## Method of Characteristics

### Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Transport equation: u_t + c*u_x = 0
# Characteristic lines: x - c*t = xi (constant along characteristics)
# Solution: u(x,t) = f(x - c*t)

def solve_transport_equation(x, t, c, initial_condition_func):
    """
    Solve u_t + c*u_x = 0 with u(x,0) = initial_condition_func(x)
    """
    X, T = np.meshgrid(x, t)
    U = initial_condition_func(X - c*T)
    return X, T, U

# Define spatial and temporal domains
x = np.linspace(-10, 10, 200)
t = np.linspace(0, 5, 100)
c = 1.0  # transport velocity

# Initial condition: Gaussian pulse
def initial_gaussian(x):
    return np.exp(-x**2)

# Solve
X, T, U = solve_transport_equation(x, t, c, initial_gaussian)

# Visualization
fig = plt.figure(figsize=(14, 10))

# 3D surface plot
ax = fig.add_subplot(2, 2, 1, projection='3d')
ax.plot_surface(X, T, U, cmap='viridis', alpha=0.8)
ax.set_xlabel('x', fontsize=10)
ax.set_ylabel('t', fontsize=10)
ax.set_zlabel('u', fontsize=10)
ax.set_title('Solution to $u_t + u_x = 0$', fontsize=11)

# Contour plot showing characteristics
ax = fig.add_subplot(2, 2, 2)
contour = ax.contourf(X, T, U, levels=20, cmap='viridis')
ax.clabel(contour, inline=True, fontsize=8)
# Plot characteristic lines
for xi in np.linspace(-10, 10, 15):
    ax.plot(xi + c*T[:, 0], T[:, 0], 'w--', alpha=0.3, linewidth=0.5)
ax.set_xlim(x[0], x[-1])
ax.set_xlabel('x', fontsize=10)
ax.set_ylabel('t', fontsize=10)
ax.set_title('Characteristics (dashed lines)', fontsize=11)
plt.colorbar(contour, ax=ax, label='u')

# Solution at different times
ax = fig.add_subplot(2, 2, 3)
for t_idx, t_val in enumerate([0, 1, 2, 3, 4]):
    idx = min(int(t_val / t[-1] * len(t)), len(t)-1)
    ax.plot(x, U[idx, :], linewidth=2,
            label=f't = {t_val:.1f}')
ax.set_xlabel('x', fontsize=10)
ax.set_ylabel('u(x,t)', fontsize=10)
ax.set_title('Solution at Different Times', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

# Nonlinear example: Burgers equation u_t + u*u_x = 0 (inviscid)
# For this, we use implicit characteristics
ax = fig.add_subplot(2, 2, 4)

# For Burgers: x - u*t = xi on characteristics
# We solve for u iteratively at each grid point

def solve_burgers_implicit(x_grid, t_grid, initial_func):
    """Solve Burgers equation using implicit characteristics"""
    U = np.zeros((len(t_grid), len(x_grid)))
    U[0, :] = initial_func(x_grid)

    for t_idx in range(1, len(t_grid)):
        t_val = t_grid[t_idx]

        for x_idx, x_val in enumerate(x_grid):
            # Solve the implicit relation x = xi + f(xi)*t for xi,
            # then u = f(xi); fixed-point iteration starting from the
            # value at the previous time level
            u_guess = U[t_idx-1, x_idx]

            for _ in range(20):  # fixed-point iteration
                xi = x_val - u_guess * t_val
                u_guess = initial_func(xi) if -10 <= xi <= 10 else 0.0

            U[t_idx, x_idx] = u_guess

    return U

# Triangular initial condition
def initial_triangle(x):
    return np.maximum(0, 1 - np.abs(x))

U_burgers = solve_burgers_implicit(x, t, initial_triangle)

im = ax.contourf(X, T, U_burgers, levels=20, cmap='viridis')
ax.set_xlabel('x', fontsize=10)
ax.set_ylabel('t', fontsize=10)
ax.set_title('Burgers Equation $u_t + u u_x = 0$', fontsize=11)
plt.colorbar(im, ax=ax, label='u')

plt.tight_layout()
plt.show()

## Finite Element Method: 1D Introduction

### Python Implementation for Poisson Equation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

# 1D Poisson equation: -u''(x) = f(x), u(0) = 0, u(1) = 0
# Weak form: integral(u'*v') = integral(f*v)

def fem_poisson_1d(n_nodes, f_func, bc_left=0, bc_right=0):
    """
    Solve -u''(x) = f(x) using piecewise linear FEM
    n_nodes: number of nodes
    f_func: forcing function
    """

    # Create uniform mesh
    x = np.linspace(0, 1, n_nodes)
    h = x[1] - x[0]

    # Assemble stiffness matrix K and load vector F
    # For piecewise linear elements: K_ij = integral(phi_i' * phi_j')
    # Local stiffness: [1, -1; -1, 1]/h
    # Local load: h/6 * [2, 1; 1, 2] * f_vals

    K = lil_matrix((n_nodes, n_nodes))
    F = np.zeros(n_nodes)

    # Element-wise assembly
    for elem in range(n_nodes - 1):
        x_left = x[elem]
        x_right = x[elem + 1]

        # Local stiffness matrix (1D case)
        K_local = np.array([[1, -1], [-1, 1]]) / h

        # Local load vector (using midpoint quadrature)
        x_mid = (x_left + x_right) / 2
        f_mid = f_func(x_mid)
        F_local = np.array([f_mid * h / 2, f_mid * h / 2])

        # Add to global matrix/vector
        K[elem, elem] += K_local[0, 0]
        K[elem, elem + 1] += K_local[0, 1]
        K[elem + 1, elem] += K_local[1, 0]
        K[elem + 1, elem + 1] += K_local[1, 1]

        F[elem] += F_local[0]
        F[elem + 1] += F_local[1]

    # Apply Dirichlet boundary conditions while K is still in LIL
    # format (cheap row modification), then convert for the solve
    K[0, :] = 0
    K[0, 0] = 1
    F[0] = bc_left

    K[-1, :] = 0
    K[-1, -1] = 1
    F[-1] = bc_right

    # Solve
    u = spsolve(K.tocsr(), F)

    return x, u

# Example 1: Constant forcing
def test_constant_forcing():
    def f_func(x):
        return 1.0

    # Analytical solution: u(x) = x(1-x)/2
    def u_analytical(x):
        return x * (1 - x) / 2

    # Solve with FEM
    x_fem, u_fem = fem_poisson_1d(51, f_func)

    # Comparison
    x_fine = np.linspace(0, 1, 200)
    u_exact = u_analytical(x_fine)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(x_fine, u_exact, 'b-', linewidth=2, label='Exact')
    ax1.plot(x_fem, u_fem, 'ro', markersize=4, label=f'FEM ({len(x_fem)} nodes)')
    ax1.set_xlabel('x', fontsize=11)
    ax1.set_ylabel('u(x)', fontsize=11)
    ax1.set_title('FEM Solution: $-u\'\'= 1$', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=10)

    # Error analysis
    u_fem_interp = np.interp(x_fine, x_fem, u_fem)
    error = u_exact - u_fem_interp
    ax2.semilogy(x_fine, np.abs(error) + 1e-15, 'g-', linewidth=2)
    ax2.set_xlabel('x', fontsize=11)
    ax2.set_ylabel('|Error|', fontsize=11)
    ax2.set_title('Absolute Error', fontsize=12)
    ax2.grid(True, alpha=0.3, which='both')

    plt.tight_layout()
    plt.show()

    print("FEM solution for constant forcing completed")

# Example 2: Variable forcing
def test_variable_forcing():
    def f_func(x):
        return 2 * np.pi**2 * np.sin(np.pi * x)

    # Analytical solution: u(x) = sin(pi*x)
    def u_analytical(x):
        return np.sin(np.pi * x)

    # Solve with different mesh sizes
    n_list = [11, 21, 51, 101]
    errors = []

    fig, ax = plt.subplots(figsize=(11, 6))

    for n in n_list:
        x_fem, u_fem = fem_poisson_1d(n, f_func)
        x_fine = np.linspace(0, 1, 500)
        u_exact = u_analytical(x_fine)
        u_fem_interp = np.interp(x_fine, x_fem, u_fem)

        error = np.max(np.abs(u_exact - u_fem_interp))
        errors.append(error)

        if n == n_list[-1]:  # Plot finest mesh solution
            ax.plot(x_fine, u_exact, 'b-', linewidth=2, label='Exact')
            ax.plot(x_fem, u_fem, 'ro', markersize=5,
                   label=f'FEM (n={n})')

    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('u(x)', fontsize=11)
    ax.set_title(r"FEM Solution: $-u'' = 2\pi^2 \sin(\pi x)$",
                fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    # Convergence study
    h_list = [1 / (n - 1) for n in n_list]
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.loglog(h_list, errors, 'ko-', linewidth=2, markersize=8,
             label='Observed error')
    ax.loglog(h_list, errors[0] * np.array(h_list)**2 / h_list[0]**2,
             'r--', linewidth=2, label='$O(h^2)$ reference')

    ax.set_xlabel('Mesh size h', fontsize=11)
    ax.set_ylabel('Max error', fontsize=11)
    ax.set_title('Convergence Analysis', fontsize=12)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    print("Convergence analysis completed")

# Run examples
test_constant_forcing()
test_variable_forcing()